# Bike Sharing Demand — Batch Scoring Validation

**Goal:** Determine whether the model can reliably predict demand for D+1 (tomorrow) and D+2 (the day after), and at what cost in accuracy compared to the 1-hour-ahead use case.

**Why this matters:** The production model was designed for next-hour forecasting, where all lag features are available. Predicting a full future day requires imputing short-term lags (1, 2, 3, 8 hours) that haven't occurred yet. This notebook quantifies the degradation introduced by that imputation and informs the final design decision for the API and dashboard.

---

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Data Loading & Feature Pipeline](#2-data-loading--feature-pipeline)
3. [Load Production Models](#3-load-production-models)
4. [Lag Imputation Strategy](#4-lag-imputation-strategy)
5. [Simulation: D+1 and D+2 Predictions](#5-simulation-d1-and-d2-predictions)
6. [Upper Bound: Model with Real Lags](#6-upper-bound-model-with-real-lags)
7. [Results & Design Decision](#7-results--design-decision)

## 1. Setup & Imports

We import production code directly from `src/` — `build_lag_features`, `build_calendar_features`, `compute_metrics`, and `FEATURES` are the same objects used in training. This ensures the validation reflects exactly what the deployed model sees.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_squared_log_error, r2_score

# Reuse production code directly
from bike_sharing.features.build_features import build_lag_features, build_calendar_features
from bike_sharing.models.train import compute_metrics, FEATURES

pd.set_option("display.max_columns", None)

## 2. Data Loading & Feature Pipeline

We load the full dataset and apply the same feature pipeline used in production. No train/val split here — we need the complete history to simulate future predictions from any point in time.

In [17]:
df = pd.read_csv("../data/raw/hour.csv")
df["dteday"]   = pd.to_datetime(df["dteday"])
df["datetime"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")

# Use production functions directly
df = build_lag_features(
    df,
    lags=[1, 2, 3, 8, 24, 48, 72, 168],
    rolling_windows=[24, 168]
)

min_date = df["dteday"].min()
df = build_calendar_features(df, drop_cols=["atemp", "yr"], min_date=min_date)

df["log_registered"] = np.log1p(df["registered"])
df["log_casual"]     = np.log1p(df["casual"])

print(f"Dataset: {df['dteday'].min().date()} → {df['dteday'].max().date()}")
print(f"Total rows: {len(df):,}")

Dataset: 2011-01-01 → 2012-12-31
Total rows: 17,379


## 3. Load Production Models

We load the trained LightGBM boosters directly from `artifacts/`. These are the exact models deployed in production — no retraining.

In [18]:
artifacts_dir = Path("../artifacts")

model_registered = lgb.Booster(model_file=str(artifacts_dir / "lgbm_registered.txt"))
model_casual     = lgb.Booster(model_file=str(artifacts_dir / "lgbm_casual.txt"))

## 4. Lag Imputation Strategy

When predicting D+1 or D+2, some lag features are unavailable because they reference hours that haven't occurred yet:

| Lag | Available for D+1? | Available for D+2? |
|---|---|---|
| `cnt_lag_1`, `cnt_lag_2`, `cnt_lag_3` | ❌ | ❌ |
| `cnt_lag_8` | ❌ | ❌ |
| `cnt_lag_24` | ✅ | ❌ |
| `cnt_lag_48` | ✅ | ✅ |
| `cnt_lag_72` | ✅ | ✅ |
| `cnt_lag_168` | ✅ | ✅ |
| `cnt_rolling_mean_24/168` | ✅ | ✅ |

**Imputation strategy:** For unavailable lags, we substitute the demand value from the same hour exactly 7 days prior. This preserves the hourly shape of the demand curve (commuter peaks, recreational patterns) better than a rolling average, which would smooth these patterns away.

The threshold for imputation is `lag <= horizon × 24` — for D+1 (horizon=1), lags ≤ 24h are imputed; for D+2 (horizon=2), lags ≤ 48h are imputed.

In [19]:
def impute_missing_lags(df: pd.DataFrame, target_date: pd.Timestamp, horizon: int) -> pd.DataFrame:
    """
    Build a feature DataFrame for a future date by imputing unavailable lags.

    For unavailable lags, instead of a 7-day average we use the exact value
    from the same hour exactly 7 days prior — a stronger proxy that preserves
    the hourly pattern rather than smoothing it out.

    Parameters
    ----------
    df : pd.DataFrame
        Full historical dataset with all lag features already computed.
    target_date : pd.Timestamp
        The date to predict (D+1 or D+2).
    horizon : int
        Number of days ahead (1 or 2).

    Returns
    -------
    pd.DataFrame
        24 rows (one per hour) with all features ready for prediction.
    """
    rows = df[df["dteday"] == target_date].copy()

    if rows.empty:
        raise ValueError(f"No data found for {target_date.date()}")

    ref_date = target_date - pd.Timedelta(days=7)
    ref_rows = df[df["dteday"] == ref_date].set_index("hr")["cnt"]

    for lag in [1, 2, 3, 8, 24, 48, 72, 168]:
        if lag <= horizon * 24:
            rows[f"cnt_lag_{lag}"] = rows["hr"].map(ref_rows)

    return rows

## 5. Simulation: D+1 and D+2 Predictions

We simulate production conditions by iterating over each day in the validation set, treating it as 'today' (D), and predicting D+1 and D+2 with imputed lags. We then compare against the actual demand values that we know from the dataset.

This is the correct evaluation methodology for future-horizon forecasting — it reflects exactly what the model would face in deployment.

In [20]:
SPLIT_RATIO = 0.8
cutoff = df["dteday"].quantile(SPLIT_RATIO)

# Use val dates as our "today" (D), predict D+1 and D+2
val_dates = df[df["dteday"] > cutoff]["dteday"].unique()

# We need D+1 and D+2 to exist in the dataset — drop last 2 days
simulation_dates = val_dates[:-2]

results = {"real_d1": [], "real_d2": [], "d1": [], "d2": []}

for d in simulation_dates:
    d1 = d + pd.Timedelta(days=1)
    d2 = d + pd.Timedelta(days=2)

    if d1 not in val_dates or d2 not in val_dates:
        continue

    for horizon, future_date, key in [(1, d1, "d1"), (2, d2, "d2")]:
        rows = impute_missing_lags(df, future_date, horizon)
        X    = rows[FEATURES]

        pred   = np.expm1(model_registered.predict(X)) + np.expm1(model_casual.predict(X))
        actual = rows["cnt"].values

        results[key].extend(pred.tolist())
        results[f"real_{key}"].extend(actual.tolist())

y_real_d1 = np.array(results["real_d1"])
y_real_d2 = np.array(results["real_d2"])
y_d1      = np.array(results["d1"])
y_d2      = np.array(results["d2"])

evaluate_batch = lambda y_true, y_pred, label: print(
    f"{label}\n"
    f"  RMSE:  {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}\n"
    f"  RMSLE: {np.sqrt(mean_squared_log_error(y_true, np.clip(y_pred, 0, None))):.4f}\n"
    f"  R²:    {r2_score(y_true, y_pred):.4f}\n"
)

evaluate_batch(y_real_d1, y_d1, "D+1 (imputed lags)")
evaluate_batch(y_real_d2, y_d2, "D+2 (imputed lags)")

D+1 (lags imputados)
  RMSE:  89.09
  RMSLE: 0.5677
  R²:    0.8364

D+2 (lags imputados)
  RMSE:  88.59
  RMSLE: 0.5670
  R²:    0.8374



## 6. Upper Bound: Model with Real Lags

To quantify the cost of imputation, we compute the model's performance on D+1 using the actual observed lag values — as if we magically knew the future. This is the theoretical ceiling that no imputation strategy can exceed.

The gap between this upper bound and the imputed results measures exactly how much information is lost by not having real short-term lags.

In [21]:
# Upper bound — model with real lags (no imputation)
results_real = {"y_true": [], "y_pred": []}

for d in simulation_dates:
    d1 = d + pd.Timedelta(days=1)
    if d1 not in val_dates:
        continue

    rows = df[df["dteday"] == d1].copy().dropna(subset=FEATURES)
    if rows.empty:
        continue

    X      = rows[FEATURES]
    pred   = np.expm1(model_registered.predict(X)) + np.expm1(model_casual.predict(X))
    actual = rows["cnt"].values

    results_real["y_true"].extend(actual.tolist())
    results_real["y_pred"].extend(pred.tolist())

y_true_real = np.array(results_real["y_true"])
y_pred_real = np.array(results_real["y_pred"])

evaluate_batch(y_true_real, y_pred_real, "Upper bound (lags reales)")
evaluate_batch(y_real_d1,   y_d1,        "D+1 (lags imputados)")
evaluate_batch(y_real_d2,   y_d2,        "D+2 (lags imputados)")

Upper bound (lags reales)
  RMSE:  45.03
  RMSLE: 0.2636
  R²:    0.9582

D+1 (lags imputados)
  RMSE:  89.09
  RMSLE: 0.5677
  R²:    0.8364

D+2 (lags imputados)
  RMSE:  88.59
  RMSLE: 0.5670
  R²:    0.8374



## 7. Results & Design Decision

### Performance summary

| Scenario | RMSE | RMSLE | R² |
|---|---|---|---|
| Upper bound (real lags) | 45.03 | 0.2636 | 0.9582 |
| D+1 (imputed lags) | 89.09 | 0.5677 | 0.8364 |
| D+2 (imputed lags) | 88.59 | 0.5670 | 0.8374 |

### Why imputation degrades performance so sharply

The ~92% increase in RMSE is not primarily caused by a poor imputation strategy — we tested both 7-day averages and exact same-hour-last-week values, and both produced similar results. The degradation is structural: **the model learned to heavily rely on short-term lags (1, 2, 3, 8 hours)** as confirmed by the MI ranking. No historical proxy can replicate the signal of "what happened in the last 3 hours", because that signal captures real-time conditions — weather changes, events, anomalies — that historical averages smooth away.

Notably, D+1 and D+2 perform almost identically despite D+2 losing `cnt_lag_24` as well. This confirms that the dominant source of degradation is the short-term lags, not the daily lag.

### Design decision

**The dashboard will be designed around next-hour forecasting** — given current conditions and recent demand history, predict demand for the next hour. This is:
- The use case the model was optimised for (RMSE 45, R² 0.96)
- Operationally meaningful — bike fleet operators make rebalancing decisions hourly
- Technically honest — no imputation, no hidden accuracy loss

D+1 batch scoring (R² 0.84) remains available as a secondary feature with an explicit accuracy disclaimer, useful for rough capacity planning rather than operational decisions.